# 040-optimised__mu_sigma — Organise records to download / convert, moment-objective selection `__mu_sigma`

Identical to nb `040` except that it reads and writes **the optimised selection folder**
`config.ss_dir(cfg, "proc_data", "gm_selection", "AvgSA_03__opt_mu_sigma")` (`data_processed/06_gm_selection/AvgSA_03_optimised__mu_sigma/`)
instead of `cfg["proc_data"]["gm_selection"]`. The original selection's lists are never written.

**Provenance guard** — `cache_utils.verify(...)` on each upstream CSV from `033-…-optimised__mu_sigma` against
the optimised `AvgSA_03_final_ensembles.pickle`. A `StaleCacheError` means re-run `033-…-optimised__mu_sigma`.

**Upstream** (from `033-…-optimised__mu_sigma`, in the optimised folder; each with a `.manifest.json`):
- `AvgSA_03_summary_esm_records_selected.csv`, `AvgSA_03_summary_ngasub_records_selected.csv`,
  `AvgSA_03_records_to_convert.csv`.

Also reads the already-downloaded state from `cfg["raw_data"]["esm_hdf5_folder"]` and
`cfg["raw_data"]["ngasub_folder"]` (shared with the original; records are physical files, so a
record already downloaded for the original selection counts as available here too).

**Output** (in the optimised folder; each stamped with a `.manifest.json`):
- `records_availability_summary.csv`, `complete_site_im_sets.csv`,
  `ngasub_rsns_to_download_by_site_completion_batched.csv` (from `summarise_availability`).
- `esm_records_to_convert.csv`, `ngasub_records_to_convert.csv` — the conversion lists that
  `042-…-optimised__mu_sigma` reads.

**Run order** — run top to bottom after `033-…-optimised__mu_sigma`.

**Scheme**: `SELECTION_CONFIG_MU_SIGMA` (moment-objective selection `__mu_sigma`; see the header of nb `032-…-optimised__mu_sigma`). The download / convert lists cover **every** stripe with a record set, including stripes that fail KS: this selection is not required to pass KS.


In [ ]:
import csv
import pandas as pd

from phd_project.scripts.cache_utils import fingerprint, verify, write_manifest
from phd_project.scripts.data_handling.summarise_record_availability import summarise_availability
from phd_project.config import config

cfg = config.load_config()
# SIDE QUEST (see side-quest_gm_selection_optimisation/README.md). This notebook belongs to
# ONE record set, pinned here - not the active study - so re-running it only ever touches
# that set's folders. GRID is the set's frozen stripe grid (a copy of the stripe IMLs as
# they were when the side quest ran), so stripes added to the main line later are ignored.
SS = "AvgSA_03__opt_mu_sigma"
GRID = config.stripe_grid_path(cfg, SS)

## Provenance guard

Verify the upstream selection CSVs are fresh before building any lists.

In [ ]:
# Provenance guard: refuse to build download/convert lists from stale selection
# CSVs. Each CSV was stamped by nb 033-optimised__mu_sigma against the OPTIMISED
# final_ensembles.pickle; verify that stamp still matches before consuming it.
# A StaleCacheError here means you must re-run nb 033-optimised__mu_sigma.

gm_selection_dir = config.ss_dir(cfg, "proc_data", "gm_selection", SS)   # NOT the original folder
selection_campaigns = ["AvgSA_03"]

for campaign in selection_campaigns:
    src = fingerprint(final_ensembles=gm_selection_dir / f"{campaign}_final_ensembles.pickle")
    for name in (f"{campaign}_summary_esm_records_selected.csv",
                 f"{campaign}_summary_ngasub_records_selected.csv",
                 f"{campaign}_records_to_convert.csv"):
        verify(gm_selection_dir / name, src)
print("Upstream selection CSVs verified fresh.")

## Build per-group download lists + NGA-Sub batchfiles

For each group (`combined`, `AvgSA_06`, `AvgSA_03`): combine the campaigns' selected records, drop those already downloaded, write a per-group NGA-Sub batchfile, and (for `combined`) the combined ESM/NGA-Sub download lists.

In [ ]:
# # --- groups to process (overlap between AvgSA_03 and AvgSA_06 is intentional) ---
groups = {
    # "combined": ["AvgSA_06", "AvgSA_03"],
    # "AvgSA_06": ["AvgSA_06"],
    "AvgSA_03": ["AvgSA_03"],
}

gm_dir = config.ss_dir(cfg, "proc_data", "gm_selection", SS)   # NOT the original folder


In [ ]:
# get a record availability summary for each site and prioritise the downloads.
# Both the input folder (where AvgSA_03_final_ensembles.pickle is looked up) and the
# output folder are the OPTIMISED selection folder.
gm_selection_dir = config.ss_dir(cfg, "proc_data", "gm_selection", SS)
ngasub_folder = cfg["raw_data"]["ngasub_folder"]
esm_folder = cfg["raw_data"]["esm_hdf5_folder"]
site_model = cfg["hazard_models"]["eshm20_wp1_site_model"]
out_dir = gm_selection_dir
_ = summarise_availability(gm_selection_dir, ngasub_folder, esm_folder, site_model, out_dir)

## Records to convert

Combine the per-campaign conversion lists into the per-database AT2/HDF5→JSON conversion lists, and report the conversion counts per group.

In [ ]:
# paths to per-campaign conversion lists:
convert_AvgSA03_fp = gm_selection_dir / f"AvgSA_03_records_to_convert.csv"

esm_convert_fp = gm_selection_dir / f"esm_records_to_convert.csv"
ngasub_convert_fp = gm_selection_dir / f"ngasub_records_to_convert.csv"

conv_03 = pd.read_csv(convert_AvgSA03_fp, dtype=str)
conv_combined = pd.concat([conv_03], ignore_index=True).drop_duplicates()

# The ESM list needs the network code (added by nb 033 on 2026-09-30): the converter uses it to
# pick the right NET.STA group when one HDF5 file holds the same station for two networks.
# Refuse an old list without it rather than silently writing a list the converter would have to
# guess from.
if "network_code" not in conv_combined.columns:
    raise KeyError(f"{convert_AvgSA03_fp.name} has no network_code column: re-run nb 033")

# provenance: the combined convert lists always derive from both campaigns' lists
convert_provenance = fingerprint(
    AvgSA_03_records_to_convert=convert_AvgSA03_fp,
    # AvgSA_06_records_to_convert=convert_AvgSA06_fp,
)

# write the combined per-database conversion lists. ESM keeps network_code; NGA-Sub records are
# identified by the RSN alone, so its list is unchanged.
columns_per_db = {
    "ESM": ["record_identifier", "component", "network_code"],
    "NGASub": ["record_identifier", "component"],
}
for db_name, fp in [("ESM", esm_convert_fp), ("NGASub", ngasub_convert_fp)]:
    sub = (conv_combined[conv_combined["database"] == db_name]
           [columns_per_db[db_name]]
           .drop_duplicates()
           .reset_index(drop=True))
    if db_name == "ESM":
        # one row per (record_identifier, component) is what keeps the JSON names unique
        # (nb 033 already checks this; repeated here because this is the list the converter reads)
        dup = sub.duplicated(["record_identifier", "component"], keep=False)
        if dup.any():
            raise ValueError(f"ESM records listed under more than one network:\n{sub[dup]}")
        if sub["network_code"].isna().any():
            raise ValueError("ESM records without a network code:\n"
                             f"{sub[sub['network_code'].isna()]}")
    sub.to_csv(fp, index=False)
    write_manifest(fp, convert_provenance)
    print(f"{fp.name}: {len(sub)} records")

# --- per-group conversion counts (reuses `groups` from cell two) ---
print()
print("Records to convert per group:")
for label, campaigns in groups.items():
    conv = pd.concat(
        [pd.read_csv(gm_dir / f"{c}_records_to_convert.csv", dtype=str) for c in campaigns],
        ignore_index=True,
    ).drop_duplicates()
    counts = {
        db: len(conv[conv["database"] == db][["record_identifier", "component"]].drop_duplicates())
        for db in ("ESM", "NGASub")
    }
    print(f"  {label:>8}: ESM {counts['ESM']:>5}   NGASub {counts['NGASub']:>5}")